# Day 10 - 1교시: EC2 인스턴스 생성

## 학습 목표

- AWS EC2의 개념과 역할 이해
- AMI(Amazon Machine Image)의 의미와 선택 기준 파악
- 인스턴스 유형, 키 페어, 보안 그룹 설정 방법 학습
- AWS 콘솔에서 EC2 인스턴스를 직접 생성할 수 있다

---

## 1. EC2란?

**EC2**(Elastic Compute Cloud): AWS에서 제공하는 가상 서버 서비스

![](https://docs.aws.amazon.com/ko_kr/AWSEC2/latest/UserGuide/images/get-started-diagram.png)

### EC2의 특징

| 특징 | 설명 |
|------|------|
| **탄력성** | 몇 분 내에 서버 생성/삭제 가능 |
| **확장성** | 필요에 따라 인스턴스 수 조절 |
| **종량제** | 사용한 시간만큼만 과금 |
| **다양한 옵션** | CPU, 메모리, 스토리지 선택 가능 |

### 왜 EC2를 사용하나요?

- 물리 서버 구매/설치 없이 즉시 서버 확보
- 트래픽에 따라 서버 수 유연하게 조절
- 다양한 OS와 소프트웨어 조합 가능
- 전 세계 리전에서 서비스 제공 가능

---
## 2. AMI (Amazon Machine Image)

**AMI**: 인스턴스를 시작하는 데 필요한 모든 소프트웨어 구성 정보를 담고 있는 가상 이미지

### 2.1 AMI의 구성 요소

![운영체제 구조](https://media.geeksforgeeks.org/wp-content/uploads/20250125094644195811/operating_system_os_.webp)


```
📦 AMI 구성

┌─────────────────────────────────────┐
│              AMI                     │
│  ┌─────────────────────────────┐   │
│  │      운영체제 (OS)           │   │
│  │   - Linux Kernel            │   │
│  │   - 시스템 라이브러리        │   │
│  └─────────────────────────────┘   │
│  ┌─────────────────────────────┐   │
│  │    사전 설치된 소프트웨어     │   │
│  │   - Python, Java 등         │   │
│  │   - 보안 패치               │   │
│  └─────────────────────────────┘   │
│  ┌─────────────────────────────┐   │
│  │       설정 정보              │   │
│  │   - 네트워크 설정           │   │
│  │   - 스토리지 매핑           │   │
│  └─────────────────────────────┘   │
└─────────────────────────────────────┘
```

### 2.2 AMI vs Docker 이미지

| 구분 | AMI | Docker 이미지 |
|------|-----|---------------|
| **범위** | OS 커널 포함 | 애플리케이션만 |
| **크기** | 수 GB ~ 수십 GB | 수 MB ~ 수 GB |
| **부팅** | 분 단위 | 초 단위 |
| **격리** | 하드웨어 수준 | 프로세스 수준 |
| **용도** | 전체 서버 환경 | 개별 앱 배포 |

**핵심 차이점**:
- AMI: OS 핵심인 **커널**까지 통째로 포함
- Docker: OS 커널을 **호스트와 공유**

> **커널**(Kernel): 프로그램과 하드웨어 사이에서 모든 요청을 중개하고
> 컴퓨터 자원(CPU, 메모리, 디스크)을 관리하는 운영체제의 핵심 부분

### 2.3 주요 AMI 종류

| AMI | 특징 | 권장 사용 |
|-----|------|----------|
| **Amazon Linux 2023** | AWS 최적화, 빠른 보안 패치 | 일반적인 AWS 워크로드 |
| **Ubuntu** | 풍부한 패키지, 넓은 커뮤니티 | 개발/테스트 환경 |
| **Deep Learning AMI** | ML 프레임워크 사전 설치 | AI/ML 워크로드 |
| **Windows Server** | Windows 애플리케이션 | .NET 애플리케이션 |

> 이번 실습에서는 **Amazon Linux 2023**을 사용합니다.
> AWS가 직접 관리하고 업데이트하며, 보안 패치가 가장 빠르고 AWS 환경에 최적화되어 있습니다.

---
## 3. 인스턴스 유형

**인스턴스 유형**: CPU, 메모리, 스토리지, 네트워크 용량의 조합

### 3.1 인스턴스 유형 명명 규칙

![](https://docs.aws.amazon.com/ko_kr/AWSEC2/latest/UserGuide/images/instance-types.png)

### 3.2 주요 인스턴스 패밀리

| 패밀리 | 특징 | 용도 |
|--------|------|------|
| **t** (범용 버스터블) | 기본 CPU + 버스트 가능 | 개발, 테스트, 소규모 앱 |
| **m** (범용) | 균형 잡힌 CPU/메모리 | 웹 서버, 앱 서버 |
| **c** (컴퓨팅 최적화) | 높은 CPU 성능 | 배치 처리, 게임 서버 |
| **r** (메모리 최적화) | 대용량 메모리 | 인메모리 DB, 캐시 |

### 3.3 실습에서 사용할 인스턴스

| 항목 | 권장 | 최소 |
|------|------|------|
| **인스턴스 유형** | t3.small | t2.micro |
| **vCPU** | 2 | 1 |
| **메모리** | 2 GB | 1 GB |
| **비용** | ~$0.02/시간 | Free Tier |

> **참고**: t2.micro는 Free Tier(월 750시간)에 포함되지만,
> Kafka 운영 시 메모리 부족이 발생할 수 있어 t3.small을 권장합니다.

### 3.4 x86 vs ARM

- **ARM**(Graviton) 인스턴스는 x86 대비 **20~40% 저렴**
- 예: t4g.small (ARM) vs t3.small (x86)
- 대부분의 Linux 워크로드에서 호환 가능

---
## 4. 키 페어 생성

**키 페어**: EC2 인스턴스에 SSH로 접속하기 위한 인증 수단

### 4.1 키 페어의 원리

```
🔐 키 페어 인증 흐름

┌─────────────────┐              ┌─────────────────┐
│   로컬 PC        │              │    EC2          │
│  ┌───────────┐  │              │  ┌───────────┐  │
│  │ 개인 키     │  │   SSH 연결    │  │ 공개 키     │  │
│  │ (.pem)    │──┼──────────────┼─▶│ (저장됨)    │  │
│  │           │  │              │  │           │  │
│  └───────────┘  │              │  └───────────┘  │
└─────────────────┘              └─────────────────┘

1. 키 페어 생성 시: 공개 키는 EC2에, 개인 키는 로컬에 저장
2. SSH 연결 시: 개인 키로 인증 → 공개 키와 매칭 확인
```

### 4.2 키 페어 옵션

| 옵션 | 설명 | 권장 |
|------|------|------|
| **RSA** | 전통적인 암호화 방식 | 호환성 필요 시 |
| **Ed25519** | 최신 암호화, 더 짧은 키 | **권장** |
| **.pem** | OpenSSH 형식 | Linux/Mac |
| **.ppk** | PuTTY 형식 | Windows (PuTTY) |

> 과거에는 RSA가 표준이었으나, 최근에는 **성능과 보안성** 면에서 **Ed25519**가 권장됩니다.

### 4.3 키 페어 생성 단계 (AWS 콘솔)

```
📋 키 페어 생성 순서

1. EC2 콘솔 → 왼쪽 메뉴 "Key Pairs"

2. "Create key pair" 클릭

3. 설정:
   ┌────────────────────────────────────────┐
   │ Name: my-ec2-key                       │
   │ Key pair type: Ed25519 (권장)          │
   │ Private key file format: .pem          │
   └────────────────────────────────────────┘

4. "Create key pair" → .pem 파일 자동 다운로드

5. 다운로드된 키 파일 안전하게 보관!
   (분실 시 인스턴스 접속 불가)
```

---
## 5. 네트워크 설정 (보안 그룹)

**보안 그룹**(Security Group): 인스턴스에 대한 인바운드/아웃바운드 트래픽을 제어하는 가상 방화벽

### 5.1 보안 그룹 개념

![](https://docs.aws.amazon.com/ko_kr/vpc/latest/userguide/images/security-group-details.png)

```
🛡️ 보안 그룹 동작

             인바운드 규칙
             (외부 → EC2)
                  │
                  ▼
┌─────────────────────────────────┐
│         Security Group          │
│  ┌───────────────────────────┐ │
│  │        EC2 Instance       │ │
│  └───────────────────────────┘ │
└─────────────────────────────────┘
                  │
                  ▼
             아웃바운드 규칙
             (EC2 → 외부)
```

- **인바운드**: 외부에서 EC2로 들어오는 트래픽 (기본: 모두 차단)
- **아웃바운드**: EC2에서 외부로 나가는 트래픽 (기본: 모두 허용)

### 5.2 실습에서 필요한 인바운드 규칙

| 유형 | 프로토콜 | 포트 | 소스 | 용도 |
|------|----------|------|------|------|
| SSH | TCP | 22 | My IP | 터미널 접속 |

> **My IP**: 현재 내 컴퓨터의 공인 IP만 허용
> 보안을 위해 **0.0.0.0/0**(모든 IP)은 피하세요!

### 5.3 보안 그룹 생성 단계

```
📋 보안 그룹 생성 (인스턴스 생성 시)

1. "Network settings" 섹션에서
   [x] Create security group 선택

2. Security group name: my-ec2-sg

3. Description: Allow SSH from My IP

4. Inbound security groups rules:
   ┌──────────┬──────────┬────────┬───────────┐
   │ Type     │ Protocol │ Port   │ Source    │
   ├──────────┼──────────┼────────┼───────────┤
   │ SSH      │ TCP      │ 22     │ My IP     │
   └──────────┴──────────┴────────┴───────────┘

5. 나머지 포트는 필요할 때 추가 (다음 교시에서 진행)
```

---
## 6. 스토리지 구성

**EBS**(Elastic Block Store): EC2에 연결되는 가상 하드 디스크

### 6.1 스토리지 옵션

| 볼륨 유형 | 특징 | 용도 |
|-----------|------|------|
| **gp3** (범용 SSD) | 균형 잡힌 성능/비용 | 일반적인 워크로드 |
| **gp2** (이전 세대) | gp3 이전 버전 | 레거시 호환 |
| **io1/io2** (프로비저닝 IOPS) | 고성능, 높은 비용 | 데이터베이스 |
| **st1** (처리량 최적화 HDD) | 대용량, 순차 읽기 | 빅데이터 |

### 6.2 실습 권장 설정

| 항목 | 설정 | 설명 |
|------|------|------|
| **크기** | 8 GiB | OS + 기본 패키지 설치 공간 |
| **볼륨 유형** | gp3 | 비용 효율적인 SSD |
| **IOPS** | 3000 (기본) | gp3 기본값 |
| **처리량** | 125 MB/s (기본) | gp3 기본값 |

> **Free Tier**: 30GB의 EBS 스토리지(gp2/gp3) 무료 제공

---
## 7. EC2 인스턴스 생성 실습

### 7.1 인스턴스 생성 전체 흐름

```
📋 EC2 인스턴스 생성 단계

AWS 콘솔 → EC2 → "Launch instance"

┌─────────────────────────────────────────────────────┐
│ 1. Name and tags                                    │
│    Name: my-kafka-server                            │
├─────────────────────────────────────────────────────┤
│ 2. Application and OS Images (AMI)                  │
│    Amazon Linux 2023 AMI (Free tier eligible)       │
│    Architecture: 64-bit (x86)                       │
├─────────────────────────────────────────────────────┤
│ 3. Instance type                                    │
│    t3.small (권장) 또는 t2.micro (Free tier)        │
├─────────────────────────────────────────────────────┤
│ 4. Key pair (login)                                 │
│    Select: my-ec2-key (기존) 또는 Create new       │
├─────────────────────────────────────────────────────┤
│ 5. Network settings                                 │
│    [x] Create security group                        │
│    [x] Allow SSH traffic from My IP                 │
├─────────────────────────────────────────────────────┤
│ 6. Configure storage                                │
│    8 GiB, gp3                                       │
├─────────────────────────────────────────────────────┤
│ 7. Launch instance                                  │
└─────────────────────────────────────────────────────┘
```

### 7.2 인스턴스 상태 확인

![](https://docs.aws.amazon.com/images/AWSEC2/latest/UserGuide/images/instance_lifecycle.png)

인스턴스 생성 후 상태 변화:

```
pending (시작 중) → running (실행 중)
       │                  │
       │                  ├── 접속 가능!
       │                  │
       └── 약 30초~1분 소요
```

| 상태 | 설명 | 과금 |
|------|------|------|
| **pending** | 인스턴스 시작 중 | X |
| **running** | 실행 중, 접속 가능 | O |
| **stopping** | 중지 중 | X |
| **stopped** | 중지됨 (EBS 유지) | EBS만 |
| **terminated** | 완전 삭제 | X |

### 7.3 중요 정보 확인

인스턴스 생성 후 다음 정보를 메모하세요:

| 항목 | 위치 | 용도 |
|------|------|------|
| **Public IPv4** | 인스턴스 세부 정보 | SSH 접속, 웹 접근 |
| **Instance ID** | 인스턴스 목록 | 인스턴스 식별 |
| **Security Group** | 보안 탭 | 포트 규칙 수정 |

---
## 퀴즈

### Q1. EC2 인스턴스에 SSH로 접속하기 위해 필요한 것은?

- A) 보안 그룹에서 22번 포트 오픈
- B) 키 페어(.pem 파일) 보유
- C) 인스턴스의 Public IP 주소
- D) 위 모두 필요

<details>
<summary>정답 확인</summary>

**정답: D) 위 모두 필요**

SSH 접속에는 세 가지가 모두 필요합니다:
- **보안 그룹 22번 포트**: 방화벽에서 SSH 트래픽 허용
- **키 페어**: 인증을 위한 개인 키
- **Public IP**: 접속할 대상 주소
</details>

---

### Q2. AMI(Amazon Machine Image)와 Docker 이미지의 핵심 차이점은?

<details>
<summary>정답 확인</summary>

**핵심 차이점**: OS 커널 포함 여부

- **AMI**: 운영체제 **커널까지 통째로** 포함된 완전한 서버 이미지
- **Docker 이미지**: 애플리케이션과 의존성만 포함, **OS 커널은 호스트와 공유**

따라서 AMI는 완전히 독립된 가상 머신을 만들고,
Docker는 호스트 OS 위에서 격리된 프로세스로 실행됩니다.
</details>

---

### Q3. 보안 그룹의 인바운드 규칙에서 Source를 "0.0.0.0/0"으로 설정하면 어떻게 되나요?

<details>
<summary>정답 확인</summary>

**0.0.0.0/0**: 모든 IP 주소에서 접근 허용

이는 인터넷의 **모든 사람**이 해당 포트로 접근할 수 있음을 의미합니다.

- SSH(22번 포트)에 0.0.0.0/0 설정: **매우 위험!**
  - 전 세계에서 브루트포스 공격 가능
- HTTP(80번 포트)에 0.0.0.0/0 설정: **웹 서비스라면 필요**
  - 공개 웹사이트는 모든 사용자가 접근해야 함

**권장**: SSH는 "My IP"로 제한, 웹 서비스만 필요 시 0.0.0.0/0 허용
</details>

---
## 핵심 요약

| 항목 | 내용 |
|------|------|
| **EC2** | AWS의 가상 서버, 탄력적으로 확장/축소 가능 |
| **AMI** | OS + 소프트웨어가 포함된 서버 이미지, Docker보다 무거움 |
| **인스턴스 유형** | t3.small 권장 (Kafka 운영 시) |
| **키 페어** | Ed25519 + .pem 형식 권장 |
| **보안 그룹** | 가상 방화벽, SSH는 My IP로 제한 |
| **스토리지** | 8GiB gp3 권장 |

```
🎯 이번 교시에서 배운 것:

1. EC2 인스턴스 = 클라우드의 가상 서버
2. AMI = 서버 템플릿 (OS + 소프트웨어)
3. 보안 그룹 = 가상 방화벽 (포트 제어)
4. 키 페어 = SSH 접속 인증 수단
```

**다음 교시 예고**: SSH로 EC2에 접속하고 VSCode와 연결합니다!

---


# Day 10 - 2교시: SSH 연결 및 VPC 인바운드 규칙

## 학습 목표

- SSH 프로토콜의 개념과 동작 원리 이해
- 로컬 터미널에서 EC2 인스턴스에 SSH로 접속할 수 있다
- VSCode Remote-SSH로 원격 개발 환경을 구성할 수 있다
- VPC 인바운드 규칙의 동작을 직접 실험하여 이해한다

---

## 1. SSH란?

**SSH**(Secure Shell): 네트워크 상의 다른 컴퓨터에 안전하게 접속하거나 명령을 실행하기 위한 보안 프로토콜

![](https://www.ipxo.com/app/uploads/2022/02/What-is-SSH-640x359.jpg)

### 1.1 SSH의 특징

```
🔐 SSH 연결 흐름

┌─────────────────┐                  ┌─────────────────┐
│   로컬 PC        │                  │    EC2 서버     │
│  (WSL2 Ubuntu)  │                  │ (Amazon Linux)  │
│                 │     암호화 통신   │                 │
│  ssh 클라이언트 ├─────────────────▶│  sshd 데몬      │
│                 │     :22 포트     │                 │
│  개인 키(.pem)  │                  │  공개 키        │
└─────────────────┘                  └─────────────────┘
```

| 특징 | 설명 |
|------|------|
| **암호화** | 모든 통신 내용이 암호화됨 |
| **인증** | 비밀번호 또는 키 기반 인증 |
| **포트** | 기본 22번 포트 사용 |
| **용도** | 원격 서버 접속, 파일 전송(SCP), 터널링 |

---
## 2. SSH 키 설정

### 2.1 키 파일 저장 위치

SSH 키 파일은 `~/.ssh/` 디렉토리에 보관합니다.

```bash
# WSL2 Ubuntu에서 실행

# .ssh 디렉토리 확인 (없으면 생성)
ls -la ~/.ssh/

# 없으면 생성
mkdir -p ~/.ssh
```

### 2.2 키 파일 복사

다운로드한 `.pem` 파일을 `~/.ssh/`로 복사합니다.

```bash
# Windows 다운로드 폴더에서 WSL로 복사
cp /mnt/c/Users/[윈도우사용자명]/Downloads/my-ec2-key.pem ~/.ssh/

# 또는 직접 이동
mv /mnt/c/Users/[윈도우사용자명]/Downloads/my-ec2-key.pem ~/.ssh/
```

### 2.3 키 파일 권한 설정

**중요!** SSH 키 파일은 본인만 읽을 수 있어야 합니다.

```bash
# 권한 변경 (소유자만 읽기 가능)
chmod 400 ~/.ssh/my-ec2-key.pem

# 권한 확인
ls -la ~/.ssh/my-ec2-key.pem
# -r--------  1 user user  1234 Jan 16 10:00 my-ec2-key.pem
```

> **chmod 400 의미**:
> - 소유자: 읽기만 가능 (4)
> - 그룹: 권한 없음 (0)
> - 기타: 권한 없음 (0)
>
> 다른 서드파티 앱이나 프로세스들이 함부로 키를 탈취하지 못하도록 설정

---
## 3. SSH 연결하기

### 3.1 기본 SSH 명령어

```bash
# SSH 연결 기본 형식
ssh -i [키 파일 경로] [사용자]@[EC2 Public IP]

# Amazon Linux 2023 접속 예시
ssh -i ~/.ssh/my-ec2-key.pem ec2-user@13.125.xxx.xxx

# Ubuntu AMI 사용 시
ssh -i ~/.ssh/my-ec2-key.pem ubuntu@13.125.xxx.xxx
```

### 3.2 첫 접속 시 fingerprint 확인

처음 접속하면 다음과 같은 메시지가 나타납니다:

```
The authenticity of host '13.125.xxx.xxx' can't be established.
ED25519 key fingerprint is SHA256:xxxxxxxxxxxx.
Are you sure you want to continue connecting (yes/no/[fingerprint])?
```

`yes`를 입력하면 해당 서버가 `~/.ssh/known_hosts`에 등록됩니다.

### 3.3 접속 성공 확인

정상 접속되면 다음과 같은 프롬프트가 표시됩니다:

```
   ,     #_
   ~\_  ####_        Amazon Linux 2023
  ~~  \_#####\
  ~~     \###|
  ~~       \#/ ___   https://aws.amazon.com/linux/amazon-linux-2023
   ~~       V~' '->
    ~~~         /
      ~~._.   _/
         _/ _/
       _/m/'
[ec2-user@ip-172-31-xx-xx ~]$
```

---
## 4. SSH Config 설정 (편리한 접속)

매번 긴 명령어를 입력하는 대신, SSH config 파일을 설정하면 간단하게 접속할 수 있습니다.

### 4.1 Config 파일 생성/수정

```bash
# ~/.ssh/config 파일 편집
nano ~/.ssh/config
```

다음 내용을 추가합니다:

```
# ~/.ssh/config

Host my-ec2
    HostName 13.125.xxx.xxx      # EC2 Public IP
    User ec2-user                 # Amazon Linux: ec2-user, Ubuntu: ubuntu
    IdentityFile ~/.ssh/my-ec2-key.pem
    StrictHostKeyChecking no      # 첫 접속 시 확인 생략 (선택)
```

### 4.2 간편 접속

이제 다음 명령어로 간단히 접속할 수 있습니다:

```bash
# 긴 명령어 대신
ssh my-ec2

# 위 명령어는 아래와 동일
ssh -i ~/.ssh/my-ec2-key.pem ec2-user@13.125.xxx.xxx
```

---
## 5. VSCode Remote-SSH 연결

Visual Studio Code에서 EC2에 직접 연결하여 개발할 수 있습니다.

### 5.1 확장 프로그램 설치

```
📋 VSCode 확장 설치

1. VSCode 실행
2. 왼쪽 사이드바 → Extensions (Ctrl+Shift+X)
3. 검색: "Remote - SSH"
4. Microsoft의 "Remote - SSH" 설치
```

### 5.2 SSH 호스트 추가

```
📋 Remote-SSH 연결 설정

1. F1 (또는 Ctrl+Shift+P) → 명령 팔레트 열기

2. "Remote-SSH: Connect to Host..." 검색 및 선택

3. "Configure SSH Hosts..." 선택

4. ~/.ssh/config 파일 선택

5. 위에서 작성한 config가 있으면 "my-ec2" 호스트 표시됨

6. "my-ec2" 선택 → 새 VSCode 창에서 EC2 연결!
```

### 5.3 연결 후 확인

- 왼쪽 하단에 **"SSH: my-ec2"** 표시
- 터미널 열기 (Ctrl+`) → EC2 터미널 사용 가능
- 파일 탐색기에서 EC2의 파일 시스템 접근

---
## 6. VPC 인바운드 규칙 이해하기

이제 보안 그룹의 인바운드 규칙이 어떻게 동작하는지 직접 실험해봅니다.

### 6.1 실험 시나리오

```
🧪 실험 목표: "문이 닫혀 있음을 확인하고, 열어보기"

1단계: EC2에서 웹 서버 실행
       ↓
2단계: 로컬 브라우저에서 접속 시도 → 실패 (포트 막힘)
       ↓
3단계: Security Group에서 포트 열기
       ↓
4단계: 다시 접속 → 성공!
```

### 6.2 EC2에서 임시 웹 서버 실행

SSH로 EC2에 접속한 후 Python 내장 웹 서버를 실행합니다.

```bash
# EC2 터미널에서 실행
sudo python3 -m http.server 80
```

> **왜 sudo가 필요한가요?**
>
> 0~1023번 포트는 **Privileged Ports**(권한 포트)라고 합니다.
> 루트 권한 없이는 이 범위의 포트를 열 수 없습니다.
> - 80: HTTP
> - 443: HTTPS
> - 22: SSH

### 6.3 python3 -m http.server 80 명령어 분석

| 요소 | 설명 |
|------|------|
| `python3` | Python 3 인터프리터 실행 |
| `-m` | 라이브러리 모듈을 스크립트처럼 실행 |
| `http.server` | Python 내장 간이 웹 서버 모듈 |
| `80` | 서버가 대기할 포트 번호 (생략 시 8000) |

**용도**:
- 개발 중 정적 파일 확인
- 서버 간 파일 전송
- **네트워크 인바운드 규칙 테스트** (지금!)

### 6.4 외부 접속 시도 (실패 예상)

로컬 PC의 웹 브라우저에서 접속합니다:

```
http://[EC2-Public-IP]
```

**예상 결과**: 무한 로딩 또는 Timeout 발생

```
❌ 현재 상태

   [로컬 브라우저]              [EC2]
   ┌──────────┐               ┌──────────────────┐
   │  요청    │──── 80 ──────▶│  Security Group  │
   │          │               │      ╳ 차단!     │
   │          │               │  ┌────────────┐  │
   │          │               │  │ Web Server │  │
   │          │               │  │  (80 포트) │  │
   │          │               │  └────────────┘  │
   └──────────┘               └──────────────────┘

   서버는 실행 중이지만, 방화벽(Security Group)이 막고 있음!
```

### 6.5 Security Group 수정 (포트 열기)

AWS 콘솔에서 HTTP 포트를 엽니다.

```
📋 인바운드 규칙 추가

1. AWS 콘솔 → EC2 → Instances → 인스턴스 선택

2. 아래 "Security" 탭 클릭

3. Security group 링크 클릭

4. "Inbound rules" 탭 → "Edit inbound rules"

5. "Add rule" 클릭:
   ┌──────────┬──────────┬────────┬───────────┐
   │ Type     │ Protocol │ Port   │ Source    │
   ├──────────┼──────────┼────────┼───────────┤
   │ HTTP     │ TCP      │ 80     │ My IP     │
   └──────────┴──────────┴────────┴───────────┘

6. "Save rules"
```

### 6.6 다시 접속 (성공!)

브라우저에서 다시 접속합니다:

```
http://[EC2-Public-IP]
```

**결과**: EC2의 현재 디렉토리 파일 목록이 표시됩니다!

```
✅ Security Group 수정 후

   [로컬 브라우저]              [EC2]
   ┌──────────┐               ┌──────────────────┐
   │  요청    │──── 80 ──────▶│  Security Group  │
   │          │               │      ✓ 허용!     │
   │  ◀────── │ 응답          │  ┌────────────┐  │
   │          │               │  │ Web Server │  │
   │          │               │  │  (80 포트) │  │
   │          │               │  └────────────┘  │
   └──────────┘               └──────────────────┘
```

### 6.7 실습 정리

EC2 터미널에서 Ctrl+C로 웹 서버를 종료합니다.

```bash
# Ctrl+C 입력
^C
[ec2-user@ip-xxx ~]$
```

> **핵심 교훈**:
> Security Group은 EC2의 **첫 번째 방어선**입니다.
> 서비스가 실행 중이어도, Security Group에서 포트를 열지 않으면 외부에서 접근할 수 없습니다.

---
## 퀴즈

### Q1. SSH 키 파일의 권한을 chmod 400으로 설정하는 이유는?

- A) 파일 크기를 줄이기 위해
- B) 다른 사용자나 프로세스가 키를 읽지 못하도록 보안 강화
- C) SSH 프로토콜의 필수 요구사항
- D) Windows 호환성을 위해

<details>
<summary>정답 확인</summary>

**정답: B) 다른 사용자나 프로세스가 키를 읽지 못하도록 보안 강화**

- SSH 클라이언트는 키 파일의 권한이 너무 열려있으면 보안 경고를 표시하고 연결을 거부할 수 있습니다
- chmod 400은 소유자만 읽기 가능하게 설정하여 다른 프로세스의 접근을 차단합니다
</details>

---

### Q2. EC2에서 python3 -m http.server 80을 실행할 때 sudo가 필요한 이유는?

<details>
<summary>정답 확인</summary>

0~1023번 포트는 **Privileged Ports**(권한 포트)입니다.

- 시스템 서비스용으로 예약된 포트 범위
- 일반 사용자는 이 범위의 포트를 열 수 없음
- 80(HTTP), 443(HTTPS), 22(SSH) 등이 포함됨

따라서 80번 포트에서 서버를 실행하려면 루트 권한(sudo)이 필요합니다.
</details>

---

### Q3. Security Group에서 인바운드 규칙을 추가하지 않으면 어떤 일이 발생하나요?

<details>
<summary>정답 확인</summary>

**외부에서 해당 포트로 접근할 수 없습니다.**

- EC2 내부에서 서비스가 정상 실행 중이어도
- Security Group이 "문을 닫고 있으면"
- 외부 요청은 EC2에 도달하지 못함 (Timeout 발생)

Security Group의 기본 인바운드 정책은 **모두 거부(Deny All)**입니다.
필요한 포트만 명시적으로 허용해야 합니다.
</details>

---
## 핵심 요약

| 항목 | 내용 |
|------|------|
| **SSH 키 위치** | ~/.ssh/my-ec2-key.pem |
| **권한 설정** | chmod 400 (소유자만 읽기) |
| **SSH 명령어** | ssh -i [키] [사용자]@[IP] |
| **SSH Config** | ~/.ssh/config에 호스트 정보 저장 |
| **VSCode** | Remote-SSH 확장으로 원격 개발 |
| **인바운드 규칙** | Security Group에서 포트 열어야 외부 접근 가능 |

```
🎯 이번 교시에서 배운 것:

1. SSH로 EC2에 안전하게 접속하는 방법
2. SSH Config로 편리하게 접속 설정
3. VSCode에서 직접 EC2 개발 환경 구성
4. Security Group = 클라우드의 방화벽 (포트 제어)
```

**다음 교시 예고**: Flask로 간단한 API 서버를 만들어봅니다!

---


# Day 10 - 3교시: Flask API 서버 만들기

## 학습 목표

- Flask 웹 프레임워크의 기본 개념 이해
- HTTP 메서드(GET, POST, PUT, DELETE)의 역할 파악
- REST API의 기본 라우팅 구조 학습
- EC2에서 간단한 API 서버를 직접 구현할 수 있다

---

> 이번 교시는 **과제 형태**로 진행됩니다.
> 각 단계별로 힌트와 모범답안이 제공되니, 먼저 직접 시도해보세요!

## 1. Flask 소개

**Flask**: Python으로 작성된 경량 웹 프레임워크

### 1.1 Flask의 특징

| 특징 | 설명 |
|------|------|
| **경량** | 핵심 기능만 제공, 필요한 것만 추가 |
| **유연성** | 프로젝트 구조 자유롭게 설계 |
| **간단함** | 몇 줄의 코드로 웹 서버 실행 |
| **확장성** | 다양한 확장 라이브러리 지원 |

### 1.2 Flask 설치

EC2에 SSH로 접속한 후 Flask를 설치합니다.

```bash
# EC2에서 실행
pip3 install flask
```

> **참고**: Amazon Linux 2023에는 Python 3가 기본 설치되어 있습니다.

---
## 2. HTTP 메서드 이해하기

### 2.1 HTTP란?

**HTTP**(HyperText Transfer Protocol): 웹에서 클라이언트와 서버가 통신하는 규약

```
📊 HTTP 요청/응답 흐름

┌─────────────┐                      ┌─────────────┐
│   클라이언트 │                      │    서버     │
│  (브라우저)  │                      │  (Flask)    │
│             │   HTTP 요청           │             │
│             │  ─────────────────▶   │             │
│             │   GET /api/users     │             │
│             │                      │             │
│             │   HTTP 응답           │             │
│             │  ◀─────────────────   │             │
│             │   200 OK + JSON      │             │
└─────────────┘                      └─────────────┘
```

### 2.2 주요 HTTP 메서드

| 메서드 | 용도 | 설명 |
|--------|------|------|
| **GET** | 조회 | 리소스를 가져올 때 (데이터 변경 X) |
| **POST** | 생성 | 새로운 리소스를 만들 때 |
| **PUT** | 수정 | 기존 리소스를 업데이트할 때 |
| **DELETE** | 삭제 | 리소스를 삭제할 때 |

### 2.3 CRUD와 HTTP 메서드 매핑

```
CRUD (데이터베이스)     HTTP 메서드 (API)
─────────────────     ─────────────────
     Create     ─────▶     POST
     Read       ─────▶     GET
     Update     ─────▶     PUT
     Delete     ─────▶     DELETE
```

---
## 3. 과제: Flask API 서버 구현

### 3.1 구현할 API 목록

| 엔드포인트 | 메서드 | 설명 | 학습 포인트 |
|-----------|--------|------|-------------|
| `/` | GET | HTML 또는 환영 메시지 반환 | 기본 라우팅 |
| `/hello` | GET | 단순 텍스트 반환 | 문자열 응답 |
| `/hello/<name>` | GET | 동적 URL 파라미터 | URL 변수 처리 |
| `/api/users` | GET | 사용자 목록 JSON | jsonify 사용 |
| `/api/users` | POST | 사용자 추가 | request.json 파싱 |
| `/api/users/<id>` | PUT | 사용자 수정 | PUT 메서드 처리 |
| `/api/users/<id>` | DELETE | 사용자 삭제 | DELETE 메서드 처리 |

---
### 과제 1: 기본 Flask 앱 만들기 (난이도: ★☆☆)

**목표**: Flask 앱을 생성하고 `/`와 `/hello` 엔드포인트를 구현하세요.

**요구사항**:
1. `/` 접속 시 "Welcome to Flask API Server!" 반환
2. `/hello` 접속 시 "Hello, World!" 반환
3. 80번 포트에서 서버 실행 (모든 IP에서 접근 가능하도록)

<details>
<summary>힌트</summary>

```python
from flask import Flask

app = Flask(__name__)

@app.route('/')  # 데코레이터로 URL 경로 지정
def index():
    return "반환할 문자열"

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=80)  # 모든 IP에서 접근 허용
```
</details>

<details>
<summary>모범답안</summary>

```python
# app.py
from flask import Flask

app = Flask(__name__)

@app.route('/')
def index():
    return "Welcome to Flask API Server!"

@app.route('/hello')
def hello():
    return "Hello, World!"

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=80, debug=True)
```

**실행 방법**:
```bash
sudo python3 app.py
```

**테스트**:
- 브라우저: `http://[EC2-IP]/`
- 브라우저: `http://[EC2-IP]/hello`
</details>

---
### 과제 2: 동적 URL 파라미터 (난이도: ★☆☆)

**목표**: URL에서 이름을 받아 인사 메시지를 반환하세요.

**요구사항**:
1. `/hello/홍길동` 접속 시 "Hello, 홍길동!" 반환
2. `/hello/Alice` 접속 시 "Hello, Alice!" 반환

<details>
<summary>힌트</summary>

```python
@app.route('/hello/<name>')  # <name>이 변수로 전달됨
def hello_name(name):
    return f"Hello, {name}!"
```
</details>

<details>
<summary>모범답안</summary>

```python
@app.route('/hello/<name>')
def hello_name(name):
    """동적 URL 파라미터 처리

    <name> 부분이 함수의 인자로 전달됩니다.
    예: /hello/홍길동 → name = "홍길동"
    """
    return f"Hello, {name}!"
```

**테스트**:
- 브라우저: `http://[EC2-IP]/hello/홍길동`
- 결과: "Hello, 홍길동!"
</details>

---
### 과제 3: JSON 응답 반환 (난이도: ★★☆)

**목표**: 사용자 목록을 JSON 형식으로 반환하세요.

**요구사항**:
1. 메모리에 사용자 데이터 저장 (리스트 또는 딕셔너리)
2. `/api/users` GET 요청 시 전체 사용자 목록 반환
3. JSON 형식으로 응답

<details>
<summary>힌트</summary>

```python
from flask import Flask, jsonify

# 메모리에 저장할 데이터 (데이터베이스 대신)
users = [
    {"id": 1, "name": "홍길동", "email": "hong@example.com"},
    {"id": 2, "name": "김철수", "email": "kim@example.com"},
]

@app.route('/api/users', methods=['GET'])
def get_users():
    return jsonify(users)  # 리스트/딕셔너리를 JSON으로 변환
```
</details>

<details>
<summary>모범답안</summary>

```python
from flask import Flask, jsonify

app = Flask(__name__)

# 인메모리 데이터 저장소
users = [
    {"id": 1, "name": "홍길동", "email": "hong@example.com"},
    {"id": 2, "name": "김철수", "email": "kim@example.com"},
    {"id": 3, "name": "이영희", "email": "lee@example.com"},
]

@app.route('/api/users', methods=['GET'])
def get_users():
    """전체 사용자 목록 조회

    jsonify(): Python 객체를 JSON 응답으로 변환
    - Content-Type: application/json 자동 설정
    - 한글도 정상 처리
    """
    return jsonify(users)
```

**테스트**:
```bash
curl http://[EC2-IP]/api/users
```

**응답 예시**:
```json
[
  {"id": 1, "name": "홍길동", "email": "hong@example.com"},
  {"id": 2, "name": "김철수", "email": "kim@example.com"},
  {"id": 3, "name": "이영희", "email": "lee@example.com"}
]
```
</details>

---
### 과제 4: POST로 사용자 추가 (난이도: ★★☆)

**목표**: POST 요청으로 새로운 사용자를 추가하세요.

**요구사항**:
1. `/api/users` POST 요청 처리
2. JSON 요청 본문에서 name, email 추출
3. 새 사용자 추가 후 생성된 사용자 정보 반환
4. HTTP 상태 코드 201 (Created) 반환

<details>
<summary>힌트</summary>

```python
from flask import Flask, jsonify, request

@app.route('/api/users', methods=['POST'])
def create_user():
    data = request.json  # JSON 요청 본문 파싱
    name = data.get('name')
    email = data.get('email')

    new_user = {
        "id": len(users) + 1,  # 간단한 ID 생성
        "name": name,
        "email": email
    }
    users.append(new_user)

    return jsonify(new_user), 201  # 201 Created
```
</details>

<details>
<summary>모범답안</summary>

```python
from flask import Flask, jsonify, request

@app.route('/api/users', methods=['POST'])
def create_user():
    """새 사용자 생성

    request.json: 요청 본문의 JSON을 Python 딕셔너리로 변환
    - Content-Type: application/json 헤더 필요
    """
    data = request.json

    # 요청 데이터 검증
    if not data:
        return jsonify({"error": "No data provided"}), 400

    name = data.get('name')
    email = data.get('email')

    if not name or not email:
        return jsonify({"error": "name and email are required"}), 400

    # 새 사용자 생성
    new_user = {
        "id": len(users) + 1,
        "name": name,
        "email": email
    }
    users.append(new_user)

    return jsonify(new_user), 201
```

**테스트**:
```bash
curl -X POST http://[EC2-IP]/api/users \
  -H "Content-Type: application/json" \
  -d '{"name": "박민수", "email": "park@example.com"}'
```

**응답 예시**:
```json
{"id": 4, "name": "박민수", "email": "park@example.com"}
```
</details>

---
### 과제 5: PUT/DELETE로 수정/삭제 (난이도: ★★★)

**목표**: 특정 사용자를 수정하거나 삭제하세요.

**요구사항**:
1. `/api/users/<id>` PUT 요청으로 사용자 정보 수정
2. `/api/users/<id>` DELETE 요청으로 사용자 삭제
3. 존재하지 않는 ID 요청 시 404 에러 반환

<details>
<summary>힌트</summary>

```python
@app.route('/api/users/<int:user_id>', methods=['PUT'])
def update_user(user_id):
    # user_id로 사용자 찾기
    user = next((u for u in users if u['id'] == user_id), None)
    if not user:
        return jsonify({"error": "User not found"}), 404
    # ... 수정 로직

@app.route('/api/users/<int:user_id>', methods=['DELETE'])
def delete_user(user_id):
    # user_id로 사용자 찾아서 삭제
    # ... 삭제 로직
```
</details>

<details>
<summary>모범답안</summary>

```python
@app.route('/api/users/<int:user_id>', methods=['PUT'])
def update_user(user_id):
    """사용자 정보 수정

    <int:user_id>: URL 파라미터를 정수로 변환
    next(): 조건에 맞는 첫 번째 요소 반환 (없으면 None)
    """
    user = next((u for u in users if u['id'] == user_id), None)

    if not user:
        return jsonify({"error": "User not found"}), 404

    data = request.json
    if data.get('name'):
        user['name'] = data['name']
    if data.get('email'):
        user['email'] = data['email']

    return jsonify(user)


@app.route('/api/users/<int:user_id>', methods=['DELETE'])
def delete_user(user_id):
    """사용자 삭제

    global users: 전역 변수 users를 수정하기 위해 필요
    """
    global users

    user = next((u for u in users if u['id'] == user_id), None)

    if not user:
        return jsonify({"error": "User not found"}), 404

    users = [u for u in users if u['id'] != user_id]

    return jsonify({"message": f"User {user_id} deleted"})
```

**테스트 (PUT)**:
```bash
curl -X PUT http://[EC2-IP]/api/users/1 \
  -H "Content-Type: application/json" \
  -d '{"name": "홍길동(수정)"}'
```

**테스트 (DELETE)**:
```bash
curl -X DELETE http://[EC2-IP]/api/users/1
```
</details>

---
### 과제 6: 전체 코드 통합 (난이도: ★★★)

**목표**: 위의 모든 기능을 하나의 완성된 Flask 앱으로 통합하세요.

<details>
<summary>전체 모범답안</summary>

```python
# app.py - 완성된 Flask API 서버
from flask import Flask, jsonify, request

app = Flask(__name__)

# 인메모리 데이터 저장소
users = [
    {"id": 1, "name": "홍길동", "email": "hong@example.com"},
    {"id": 2, "name": "김철수", "email": "kim@example.com"},
    {"id": 3, "name": "이영희", "email": "lee@example.com"},
]


# ============================================
# 기본 라우팅
# ============================================
@app.route('/')
def index():
    return "Welcome to Flask API Server!"


@app.route('/hello')
def hello():
    return "Hello, World!"


@app.route('/hello/<name>')
def hello_name(name):
    return f"Hello, {name}!"


# ============================================
# Users API (CRUD)
# ============================================
@app.route('/api/users', methods=['GET'])
def get_users():
    """전체 사용자 목록 조회"""
    return jsonify(users)


@app.route('/api/users/<int:user_id>', methods=['GET'])
def get_user(user_id):
    """특정 사용자 조회"""
    user = next((u for u in users if u['id'] == user_id), None)
    if not user:
        return jsonify({"error": "User not found"}), 404
    return jsonify(user)


@app.route('/api/users', methods=['POST'])
def create_user():
    """새 사용자 생성"""
    data = request.json

    if not data:
        return jsonify({"error": "No data provided"}), 400

    name = data.get('name')
    email = data.get('email')

    if not name or not email:
        return jsonify({"error": "name and email are required"}), 400

    new_user = {
        "id": max(u['id'] for u in users) + 1 if users else 1,
        "name": name,
        "email": email
    }
    users.append(new_user)

    return jsonify(new_user), 201


@app.route('/api/users/<int:user_id>', methods=['PUT'])
def update_user(user_id):
    """사용자 정보 수정"""
    user = next((u for u in users if u['id'] == user_id), None)

    if not user:
        return jsonify({"error": "User not found"}), 404

    data = request.json
    if data.get('name'):
        user['name'] = data['name']
    if data.get('email'):
        user['email'] = data['email']

    return jsonify(user)


@app.route('/api/users/<int:user_id>', methods=['DELETE'])
def delete_user(user_id):
    """사용자 삭제"""
    global users

    user = next((u for u in users if u['id'] == user_id), None)

    if not user:
        return jsonify({"error": "User not found"}), 404

    users = [u for u in users if u['id'] != user_id]

    return jsonify({"message": f"User {user_id} deleted"})


# ============================================
# 에러 핸들러
# ============================================
@app.errorhandler(404)
def not_found(e):
    return jsonify({"error": "Resource not found"}), 404


@app.errorhandler(500)
def internal_error(e):
    return jsonify({"error": "Internal server error"}), 500


if __name__ == '__main__':
    app.run(host='0.0.0.0', port=80, debug=True)
```
</details>

---
## 4. 테스트 시나리오

완성된 API를 다음 순서로 테스트해보세요.

### 4.1 브라우저 테스트

```
1. http://[EC2-IP]/            → "Welcome to Flask API Server!"
2. http://[EC2-IP]/hello       → "Hello, World!"
3. http://[EC2-IP]/hello/홍길동 → "Hello, 홍길동!"
4. http://[EC2-IP]/api/users   → JSON 사용자 목록
```

### 4.2 curl 테스트

```bash
# GET - 전체 사용자 조회
curl http://[EC2-IP]/api/users

# POST - 사용자 추가
curl -X POST http://[EC2-IP]/api/users \
  -H "Content-Type: application/json" \
  -d '{"name": "테스트", "email": "test@example.com"}'

# PUT - 사용자 수정
curl -X PUT http://[EC2-IP]/api/users/1 \
  -H "Content-Type: application/json" \
  -d '{"name": "수정된이름"}'

# DELETE - 사용자 삭제
curl -X DELETE http://[EC2-IP]/api/users/1

# GET - 삭제 확인
curl http://[EC2-IP]/api/users
```

---
## 퀴즈

### Q1. Flask에서 URL 변수를 함수 인자로 받으려면 어떻게 해야 하나요?

- A) @app.route('/user?id=<id>')
- B) @app.route('/user/<id>')
- C) @app.route('/user/{id}')
- D) @app.route('/user/:id')

<details>
<summary>정답 확인</summary>

**정답: B) @app.route('/user/<id>')**

Flask는 `<변수명>` 형식으로 URL 변수를 정의합니다.
- `<id>`: 문자열로 전달
- `<int:id>`: 정수로 변환하여 전달

다른 옵션들은 다른 프레임워크의 문법입니다:
- A) 쿼리 파라미터 형식
- C) FastAPI 스타일
- D) Express.js 스타일
</details>

---

### Q2. HTTP POST와 PUT의 차이점은?

<details>
<summary>정답 확인</summary>

| 메서드 | 용도 | 특징 |
|--------|------|------|
| **POST** | 새 리소스 생성 | 매번 새로운 리소스 생성 |
| **PUT** | 기존 리소스 수정 | 동일 요청 반복해도 결과 동일 (멱등성) |

**예시**:
- POST /api/users: 새 사용자 생성 (ID 자동 부여)
- PUT /api/users/1: ID 1번 사용자 정보 수정
</details>

---
## 핵심 요약

| 항목 | 내용 |
|------|------|
| **Flask** | Python 경량 웹 프레임워크 |
| **@app.route** | URL과 함수를 연결하는 데코레이터 |
| **jsonify** | Python 객체를 JSON 응답으로 변환 |
| **request.json** | POST/PUT 요청의 JSON 본문 파싱 |
| **HTTP 상태코드** | 200(OK), 201(Created), 404(Not Found), 400(Bad Request) |

```
🎯 이번 교시에서 배운 것:

1. Flask로 간단한 웹 서버 만들기
2. HTTP 메서드와 REST API 개념
3. JSON 요청/응답 처리
4. CRUD API 구현 패턴
```

**다음 교시 예고**: Flask 서버를 Docker로 컨테이너화합니다!

---


# Day 10 - 4교시: Docker 환경 구축 및 Flask 이식

## 학습 목표

- EC2(Amazon Linux 2023)에 Docker를 설치할 수 있다
- Docker Compose를 설치하고 사용할 수 있다
- Flask 애플리케이션을 Docker 컨테이너로 이식할 수 있다
- 컨테이너화된 앱을 EC2에서 실행하고 확인할 수 있다

---

## 1. EC2에 Docker 설치

Amazon Linux 2023에서는 `dnf` 패키지 관리자를 사용합니다.

### 1.1 Docker 설치 명령어

SSH로 EC2에 접속한 후 다음 명령어를 순서대로 실행합니다.

```bash
# 1. 시스템 패키지 업데이트
sudo dnf update -y

# 2. Docker 설치
sudo dnf install -y docker

# 3. Docker 서비스 시작
sudo systemctl start docker

# 4. 부팅 시 Docker 자동 시작 설정
sudo systemctl enable docker

# 5. 현재 사용자(ec2-user)를 docker 그룹에 추가
# (sudo 없이 docker 명령어 사용 가능)
sudo usermod -aG docker $USER

# 6. 그룹 변경 사항 바로 적용
newgrp docker
```

### 1.2 각 명령어 설명

| 명령어 | 설명 |
|--------|------|
| `dnf update -y` | 시스템 패키지 최신화 |
| `dnf install -y docker` | Docker 엔진 설치 |
| `systemctl start docker` | Docker 데몬 시작 |
| `systemctl enable docker` | 부팅 시 자동 시작 등록 |
| `usermod -aG docker $USER` | docker 그룹에 사용자 추가 |
| `newgrp docker` | 로그아웃 없이 그룹 변경 적용 |

> **참고**: `newgrp docker` 대신 SSH 재접속해도 됩니다.

---
## 2. Docker Compose 설치

Docker Compose v2는 Docker CLI 플러그인으로 설치합니다.

### 2.1 설치 명령어

```bash
# 1. Docker 플러그인 디렉토리 생성
mkdir -p ~/.docker/cli-plugins/

# 2. Docker Compose 바이너리 다운로드
curl -SL https://github.com/docker/compose/releases/download/v2.21.0/docker-compose-linux-x86_64 \
  -o ~/.docker/cli-plugins/docker-compose

# 3. 실행 권한 부여
chmod +x ~/.docker/cli-plugins/docker-compose

# 4. 설치 확인
docker compose version
```

### 2.2 설치 확인

```bash
$ docker compose version
Docker Compose version v2.21.0
```

> **참고**: Docker Compose v2부터는 `docker-compose` 대신 `docker compose` (하이픈 없이)를 사용합니다.

---
## 3. Docker 설치 확인

### 3.1 버전 확인

```bash
# Docker 버전 확인
docker --version
# Docker version 25.0.x, build xxxx

# Docker Compose 버전 확인
docker compose version
# Docker Compose version v2.21.0
```

### 3.2 테스트 컨테이너 실행

```bash
# hello-world 이미지로 테스트
docker run hello-world
```

정상 실행 시 다음과 같은 메시지가 표시됩니다:

```
Hello from Docker!
This message shows that your installation appears to be working correctly.
...
```

---
## 4. Flask 앱 도커화

이전 교시에서 만든 Flask API 서버를 Docker로 이식합니다.

### 4.1 프로젝트 구조

```bash
# 작업 디렉토리 생성
mkdir -p ~/flask-api
cd ~/flask-api
```

최종 구조:

```
~/flask-api/
├── app.py              # Flask 애플리케이션
├── requirements.txt    # Python 의존성
├── Dockerfile          # Docker 이미지 빌드 설정
└── compose.yml         # Docker Compose 설정
```

### 4.2 app.py 작성

```bash
# app.py 생성
cat > app.py << 'EOF'
from flask import Flask, jsonify, request

app = Flask(__name__)

users = [
    {"id": 1, "name": "홍길동", "email": "hong@example.com"},
    {"id": 2, "name": "김철수", "email": "kim@example.com"},
]


@app.route('/')
def index():
    return "Welcome to Flask API Server (Docker)!"


@app.route('/hello/<name>')
def hello(name):
    return f"Hello, {name}!"


@app.route('/api/users', methods=['GET'])
def get_users():
    return jsonify(users)


@app.route('/api/users', methods=['POST'])
def create_user():
    data = request.json
    if not data or not data.get('name') or not data.get('email'):
        return jsonify({"error": "name and email required"}), 400

    new_user = {
        "id": len(users) + 1,
        "name": data['name'],
        "email": data['email']
    }
    users.append(new_user)
    return jsonify(new_user), 201


if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
EOF
```

### 4.3 requirements.txt 작성

```bash
# requirements.txt 생성
cat > requirements.txt << 'EOF'
flask==3.0.0
EOF
```

---
## 5. Dockerfile 작성

### 5.1 Dockerfile 생성

```bash
cat > Dockerfile << 'EOF'
# Python 베이스 이미지
FROM python:3.11-slim

# 작업 디렉토리 설정
WORKDIR /app

# 의존성 파일 복사 및 설치
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 애플리케이션 코드 복사
COPY app.py .

# 포트 노출
EXPOSE 5000

# 실행 명령
CMD ["python", "app.py"]
EOF
```

### 5.2 Dockerfile 설명

| 명령어 | 설명 |
|--------|------|
| `FROM python:3.11-slim` | Python 3.11 경량 이미지 기반 |
| `WORKDIR /app` | 컨테이너 내 작업 디렉토리 설정 |
| `COPY requirements.txt .` | 의존성 파일 먼저 복사 (캐시 활용) |
| `RUN pip install ...` | Python 패키지 설치 |
| `COPY app.py .` | 애플리케이션 코드 복사 |
| `EXPOSE 5000` | 문서화 목적 (실제 포트 오픈 X) |
| `CMD ["python", "app.py"]` | 컨테이너 시작 시 실행할 명령 |

---
## 6. Docker Compose 설정

### 6.1 compose.yml 작성

```bash
cat > compose.yml << 'EOF'
services:
  flask-api:
    build: .
    container_name: flask-api
    ports:
      - "80:5000"
    restart: unless-stopped
EOF
```

### 6.2 compose.yml 설명

| 항목 | 설명 |
|------|------|
| `services` | 실행할 서비스 정의 |
| `build: .` | 현재 디렉토리의 Dockerfile로 이미지 빌드 |
| `container_name` | 컨테이너 이름 지정 |
| `ports: "80:5000"` | 호스트 80 → 컨테이너 5000 포트 매핑 |
| `restart: unless-stopped` | 컨테이너 자동 재시작 (수동 중지 제외) |

> **포트 매핑**: 외부에서 80번으로 접속하면 컨테이너의 5000번으로 전달

---
## 7. 빌드 및 실행

### 7.1 이미지 빌드 및 컨테이너 시작

```bash
# 현재 디렉토리 확인
cd ~/flask-api
ls -la

# 이미지 빌드 및 컨테이너 시작 (백그라운드)
docker compose up -d --build
```

### 7.2 실행 상태 확인

```bash
# 컨테이너 상태 확인
docker compose ps

# 예상 출력:
# NAME        COMMAND           STATUS          PORTS
# flask-api   "python app.py"   Up xx seconds   0.0.0.0:80->5000/tcp
```

### 7.3 로그 확인

```bash
# 실시간 로그 확인 (Ctrl+C로 종료)
docker compose logs -f flask-api
```

---
## 8. 동작 확인

### 8.1 브라우저에서 확인

웹 브라우저에서 EC2 Public IP로 접속합니다.

```
http://[EC2-Public-IP]/
→ "Welcome to Flask API Server (Docker)!"

http://[EC2-Public-IP]/hello/홍길동
→ "Hello, 홍길동!"

http://[EC2-Public-IP]/api/users
→ JSON 사용자 목록
```

### 8.2 curl로 API 테스트

```bash
# 로컬 PC에서 테스트
curl http://[EC2-Public-IP]/api/users

# POST 테스트
curl -X POST http://[EC2-Public-IP]/api/users \
  -H "Content-Type: application/json" \
  -d '{"name": "도커유저", "email": "docker@example.com"}'
```

---
## 9. 컨테이너 관리 명령어

### 9.1 자주 사용하는 명령어

```bash
# 컨테이너 중지
docker compose stop

# 컨테이너 시작 (중지된 컨테이너)
docker compose start

# 컨테이너 재시작
docker compose restart

# 컨테이너 삭제 (중지 + 삭제)
docker compose down

# 컨테이너 + 이미지까지 삭제
docker compose down --rmi all

# 컨테이너 내부 접속
docker exec -it flask-api /bin/bash
```

### 9.2 상태 확인 명령어

```bash
# 실행 중인 컨테이너 목록
docker ps

# 모든 컨테이너 목록 (중지된 것 포함)
docker ps -a

# 이미지 목록
docker images

# 디스크 사용량
docker system df
```

---
## 퀴즈

### Q1. Docker Compose v2에서 올바른 명령어 형식은?

- A) docker-compose up -d
- B) docker compose up -d
- C) docker_compose up -d
- D) dockercompose up -d

<details>
<summary>정답 확인</summary>

**정답: B) docker compose up -d**

Docker Compose v2부터는 독립 실행 파일(`docker-compose`)이 아닌
Docker CLI의 플러그인으로 동작합니다.

따라서 하이픈(-) 없이 `docker compose`로 사용합니다.
</details>

---

### Q2. compose.yml에서 ports: "80:5000"의 의미는?

<details>
<summary>정답 확인</summary>

**형식**: `호스트포트:컨테이너포트`

- **80**: EC2(호스트)에서 열리는 포트
- **5000**: 컨테이너 내부에서 Flask가 실행되는 포트

외부에서 EC2의 80번 포트로 요청하면,
Docker가 해당 요청을 컨테이너의 5000번 포트로 전달합니다.
</details>

---

### Q3. `sudo usermod -aG docker $USER` 명령어가 필요한 이유는?

<details>
<summary>정답 확인</summary>

**Docker 데몬은 root 권한으로 실행**되기 때문입니다.

기본적으로 일반 사용자는 Docker 명령어 실행 시 매번 `sudo`가 필요합니다.

`docker` 그룹에 사용자를 추가하면:
- `sudo` 없이 `docker` 명령어 사용 가능
- 개발 편의성 향상

변경 사항 적용을 위해 `newgrp docker` 또는 재접속이 필요합니다.
</details>

---
## 핵심 요약

| 항목 | 명령어/설명 |
|------|-------------|
| **Docker 설치** | `sudo dnf install -y docker` |
| **Docker 시작** | `sudo systemctl start docker` |
| **Compose 설치** | CLI 플러그인으로 설치 |
| **빌드 & 실행** | `docker compose up -d --build` |
| **상태 확인** | `docker compose ps` |
| **로그 확인** | `docker compose logs -f` |
| **중지** | `docker compose down` |

```
🎯 이번 교시에서 배운 것:

1. EC2(Amazon Linux)에 Docker/Compose 설치
2. Dockerfile로 Flask 앱 이미지 빌드
3. compose.yml로 컨테이너 관리
4. 포트 매핑으로 외부 접근 설정
```

**다음 교시 예고**: EC2에 Kafka 클러스터를 배포하고 로컬에서 연결합니다!

---


# Day 10 - 5교시: EC2에 Kafka 배포하고 로컬에서 연결하기

## 학습 목표

- EC2 인스턴스에 Docker Compose로 Kafka를 배포할 수 있다
- Security Group 설정으로 외부에서 Kafka에 접근할 수 있도록 구성한다
- KAFKA_ADVERTISED_LISTENERS 설정의 역할을 이해하고 외부 연결용 리스너를 구성한다
- 로컬(WSL2)에서 EC2의 Kafka에 연결하여 메시지를 주고받을 수 있다
- AWS 환경에서 서비스 운영의 기본 흐름을 체험한다

---

## 왜 이 실습이 중요한가요?

지금까지 우리는 로컬 환경(Docker Desktop)에서 Kafka를 사용했습니다.
하지만 실제 프로젝트에서는 AWS 같은 클라우드 환경에서 서비스를 운영합니다.

```
지금까지 (로컬 개발)                앞으로 (클라우드 운영)
┌─────────────────────┐            ┌─────────────────────┐
│     내 PC (WSL2)     │            │       AWS EC2       │
│  ┌───────────────┐  │            │  ┌───────────────┐  │
│  │    Docker     │  │            │  │    Docker     │  │
│  │    Kafka      │  │            │  │    Kafka      │  │
│  │    Producer   │  │   ────▶    │  │               │  │
│  │    Consumer   │  │            │  └───────────────┘  │
│  └───────────────┘  │            │                     │
└─────────────────────┘            └─────────────────────┘
                                             │
                                             │ 9093 포트
                                             ▼
                                   ┌─────────────────────┐
                                   │     내 PC (WSL2)     │
                                   │  Python Producer    │
                                   │  Python Consumer    │
                                   └─────────────────────┘
```

이번 실습에서는:
1. EC2에 Kafka를 배포하고
2. **네트워크 설정**(Security Group, ADVERTISED_LISTENERS)을 통해
3. 로컬에서 클라우드의 Kafka에 연결하는 경험을 합니다.

---
## 전체 실습 아키텍처

```
┌─────────────────────────────────────────────────────────────────────────┐
│                              실습 아키텍처                                │
│                                                                          │
│   [로컬 PC - WSL2 Ubuntu]              [AWS EC2 인스턴스]                │
│   ┌───────────────────┐                ┌────────────────────────────┐   │
│   │  Python Producer  │                │      Docker Compose        │   │
│   │  Python Consumer  │ ──── 9093 ────▶│  ┌────────────────────┐   │   │
│   │                   │    (Public IP) │  │  Kafka Broker      │   │   │
│   │  confluent-kafka  │                │  │  - 9092 (내부)      │   │   │
│   └───────────────────┘                │  │  - 9093 (외부)      │   │   │
│           │                            │  └────────────────────┘   │   │
│           │                            │  ┌────────────────────┐   │   │
│           └── SSH (22) ───────────────▶│  │  Kafka UI (8080)   │   │   │
│                                        │  └────────────────────┘   │   │
│                                        └────────────────────────────┘   │
│                                                     │                    │
│                                        Security Group 인바운드:          │
│                                        - 22 (SSH) from My IP            │
│                                        - 9093 (Kafka) from My IP        │
│                                        - 8080 (UI) from My IP           │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## Part 1: Security Group 설정

EC2에서 Kafka를 운영하려면 **3개의 포트**를 추가로 열어야 합니다.

### 1.1 필요한 인바운드 규칙

| 유형 | 프로토콜 | 포트 | 소스 | 용도 |
|------|----------|------|------|------|
| SSH | TCP | 22 | My IP | EC2 접속 (기존) |
| Custom TCP | TCP | 9093 | My IP | Kafka 외부 연결 |
| Custom TCP | TCP | 8080 | My IP | Kafka UI 웹 접근 |

### 1.2 Security Group 수정 (AWS 콘솔)

```
📋 인바운드 규칙 추가

1. EC2 콘솔 → Instances → 인스턴스 선택

2. 아래 "Security" 탭 → Security group 클릭

3. "Inbound rules" 탭 → "Edit inbound rules"

4. "Add rule" 클릭하여 다음 규칙 추가:
   ┌──────────────┬──────────┬────────┬─────────────┐
   │ Type         │ Protocol │ Port   │ Source      │
   ├──────────────┼──────────┼────────┼─────────────┤
   │ Custom TCP   │ TCP      │ 9093   │ My IP       │
   │ Custom TCP   │ TCP      │ 8080   │ My IP       │
   └──────────────┴──────────┴────────┴─────────────┘

5. "Save rules"
```

---
## Part 2: EC2에 Kafka 배포

### 2.1 Kafka 디렉토리 생성

SSH로 EC2에 접속한 후 작업 디렉토리를 생성합니다.

```bash
# 작업 디렉토리 생성
mkdir -p ~/kafka-ec2
cd ~/kafka-ec2
```

### 2.2 compose.yml 작성

**핵심은 KAFKA_ADVERTISED_LISTENERS 설정**입니다.

```bash
cat > compose.yml << 'EOF'
services:
  kafka:
    image: apache/kafka:latest
    container_name: kafka-broker
    ports:
      - "9092:9092"   # 내부용 (EC2 안에서만)
      - "9093:9093"   # 외부용 (로컬 PC에서 접근)
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka:9094

      # ============================================
      # 리스너 설정 (핵심!)
      # ============================================
      # LISTENERS: Kafka가 실제로 바인딩하는 주소
      KAFKA_LISTENERS: INTERNAL://0.0.0.0:9092,EXTERNAL://0.0.0.0:9093,CONTROLLER://0.0.0.0:9094

      # ADVERTISED_LISTENERS: 클라이언트에게 알려줄 주소
      # - INTERNAL: Docker 내부 통신용 (컨테이너 이름)
      # - EXTERNAL: 외부(로컬 PC)에서 접근용 (EC2 Public IP)
      KAFKA_ADVERTISED_LISTENERS: INTERNAL://kafka:9092,EXTERNAL://${EC2_PUBLIC_IP}:9093

      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: INTERNAL:PLAINTEXT,EXTERNAL:PLAINTEXT,CONTROLLER:PLAINTEXT
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_INTER_BROKER_LISTENER_NAME: INTERNAL

      # 토픽 자동 생성 및 기본 설정
      KAFKA_AUTO_CREATE_TOPICS_ENABLE: "true"
      KAFKA_NUM_PARTITIONS: 3
      KAFKA_DEFAULT_REPLICATION_FACTOR: 1
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
    healthcheck:
      test: ["CMD-SHELL", "/opt/kafka/bin/kafka-broker-api-versions.sh --bootstrap-server localhost:9092 || exit 1"]
      interval: 10s
      timeout: 10s
      retries: 5
      start_period: 30s

  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    container_name: kafka-ui
    ports:
      - "8080:8080"
    depends_on:
      kafka:
        condition: service_healthy
    environment:
      KAFKA_CLUSTERS_0_NAME: ec2-kafka
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: kafka:9092
EOF
```

---
## Part 3: KAFKA_ADVERTISED_LISTENERS 이해하기

이 설정이 왜 중요한지 그림으로 이해해봅시다.

### 3.1 문제 상황 (EXTERNAL 설정 없을 때)

```
❌ ADVERTISED_LISTENERS가 kafka:9092만 있을 때

   [로컬 PC]                          [EC2]
   ┌──────────┐                      ┌──────────────────┐
   │ Producer │ ──── 1. 연결 ───────▶ │ Kafka (9093)     │
   │          │                      │                  │
   │          │◀── 2. 응답 ───────────│ "다음에는          │
   │          │    "kafka:9092로 와"  │  kafka:9092로 와" │
   │          │                      └──────────────────┘
   │          │
   │          │ ──── 3. 재연결 시도 ──▶ kafka:9092 ???
   │          │                         (알 수 없는 호스트!)
   └──────────┘

   결과: 첫 연결은 되지만, 메타데이터 갱신 시 연결 끊김!
```

### 3.2 해결책 (EXTERNAL 리스너 분리)

```
✅ EXTERNAL을 EC2 Public IP로 설정

   [로컬 PC]                          [EC2]
   ┌──────────┐                      ┌──────────────────┐
   │ Producer │ ──── 1. 연결 ────────▶│ Kafka (9093)     │
   │          │      (EXTERNAL)      │                  │
   │          │◀── 2. 응답 ───────────│ "다음에도          │
   │          │    "EC2_IP:9093으로"  │  EC2_IP:9093으로" │
   │          │                      └──────────────────┘
   │          │
   │          │ ──── 3. 재연결 ──────▶ EC2_IP:9093 ✅
   │          │                         (연결 유지!)
   └──────────┘

   ADVERTISED_LISTENERS 설정:
   - INTERNAL://kafka:9092     → 컨테이너 내부용
   - EXTERNAL://EC2_IP:9093    → 외부(로컬)용
```

### 3.3 핵심 포인트

- Kafka 클라이언트는 첫 연결 후 브로커에게 **"다음 연결 주소"**를 받습니다
- 이 주소가 `ADVERTISED_LISTENERS`입니다
- 외부에서 접속할 때는 **EC2의 Public IP**를 알려줘야 합니다

---
## Part 4: Kafka 실행

### 4.1 EC2 Public IP 환경 변수 설정

```bash
# EC2에서 실행
cd ~/kafka-ec2

# EC2 Public IP 환경 변수 설정 (중요!)
export EC2_PUBLIC_IP=$(curl -s http://169.254.169.254/latest/meta-data/public-ipv4)
echo "EC2 Public IP: $EC2_PUBLIC_IP"
```

> **169.254.169.254**: AWS 메타데이터 서비스 IP
> EC2 인스턴스 내부에서만 접근 가능하며, 인스턴스 정보를 제공합니다.

### 4.2 Kafka 시작

```bash
# Kafka 시작
docker compose up -d

# 상태 확인
docker compose ps

# 로그 확인 (Ctrl+C로 종료)
docker compose logs -f kafka
```

### 4.3 정상 실행 확인

```
NAME           STATUS          PORTS
kafka-broker   Up (healthy)    0.0.0.0:9092->9092/tcp, 0.0.0.0:9093->9093/tcp
kafka-ui       Up              0.0.0.0:8080->8080/tcp
```

### 4.4 Kafka UI 접속 확인

브라우저에서 접속:
```
http://[EC2_PUBLIC_IP]:8080
```

"ec2-kafka" 클러스터가 보이면 성공!

---
## Part 5: 로컬에서 EC2 Kafka 연결 테스트

이제 로컬 PC(WSL2)에서 EC2의 Kafka에 연결해봅니다.

### 5.1 로컬 환경 준비

```bash
# WSL2 Ubuntu에서
pip install confluent-kafka
```

### 5.2 Producer 코드 (로컬에서 실행)

In [ ]:
# local_producer.py
# 이 코드를 로컬 PC(WSL2)에서 실행합니다

from confluent_kafka import Producer
import json

# =============================================================================
# EC2 Public IP 설정 (본인의 EC2 IP로 변경!)
# =============================================================================
EC2_PUBLIC_IP = "YOUR_EC2_PUBLIC_IP"  # 예: "13.125.xxx.xxx"
KAFKA_BOOTSTRAP_SERVERS = f"{EC2_PUBLIC_IP}:9093"


def delivery_callback(err, msg):
    """메시지 전송 결과 콜백"""
    if err:
        print(f"  전송 실패: {err}")
    else:
        print(f"  전송 성공: partition={msg.partition()}, offset={msg.offset()}")


# Producer 설정
config = {
    "bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
    "client.id": "local-producer",
}

producer = Producer(config)

print(f"EC2 Kafka에 연결 중... ({KAFKA_BOOTSTRAP_SERVERS})")
print("-" * 50)

# 테스트 메시지 전송
for i in range(1, 6):
    message = {
        "message_id": i,
        "content": f"Hello from Local PC #{i}",
        "source": "WSL2 Ubuntu",
    }

    producer.produce(
        topic="test-from-local",
        key=f"msg-{i}".encode("utf-8"),
        value=json.dumps(message, ensure_ascii=False).encode("utf-8"),
        callback=delivery_callback,
    )
    print(f"메시지 {i} 버퍼에 추가")

# 모든 메시지 전송 완료 대기
producer.flush()
print("-" * 50)
print("모든 메시지 전송 완료!")
print(f"Kafka UI 확인: http://{EC2_PUBLIC_IP}:8080")

### 5.3 Consumer 코드 (로컬에서 실행)

In [ ]:
# local_consumer.py
# 이 코드를 로컬 PC(WSL2)에서 실행합니다

from confluent_kafka import Consumer, KafkaError
import json
import signal

# =============================================================================
# EC2 Public IP 설정 (본인의 EC2 IP로 변경!)
# =============================================================================
EC2_PUBLIC_IP = "YOUR_EC2_PUBLIC_IP"  # 예: "13.125.xxx.xxx"
KAFKA_BOOTSTRAP_SERVERS = f"{EC2_PUBLIC_IP}:9093"

# Consumer 설정
config = {
    "bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
    "group.id": "local-consumer-group",
    "client.id": "local-consumer",
    "auto.offset.reset": "earliest",  # 처음부터 읽기
    "enable.auto.commit": True,
}

consumer = Consumer(config)
running = True


def signal_handler(signum, frame):
    """Ctrl+C 처리"""
    global running
    print("\n종료 신호 수신...")
    running = False


signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

print(f"EC2 Kafka Consumer 시작 ({KAFKA_BOOTSTRAP_SERVERS})")
print("토픽: test-from-local")
print("Ctrl+C로 종료")
print("-" * 50)

consumer.subscribe(["test-from-local"])

try:
    while running:
        msg = consumer.poll(timeout=1.0)

        if msg is None:
            continue

        if msg.error():
            if msg.error().code() != KafkaError._PARTITION_EOF:
                print(f"에러: {msg.error()}")
            continue

        # 메시지 처리
        key = msg.key().decode("utf-8") if msg.key() else None
        value = json.loads(msg.value().decode("utf-8"))

        print(f"메시지 수신:")
        print(f"  Key: {key}")
        print(f"  Value: {value}")
        print(f"  Partition: {msg.partition()}, Offset: {msg.offset()}")
        print()

finally:
    consumer.close()
    print("Consumer 종료 완료")

### 5.4 실행 순서

터미널 2개를 열어서 테스트합니다:

```bash
# 터미널 1: Consumer 먼저 실행 (메시지 대기)
python local_consumer.py

# 터미널 2: Producer 실행 (메시지 전송)
python local_producer.py
```

**예상 출력 (Consumer)**:
```
EC2 Kafka Consumer 시작 (13.125.xxx.xxx:9093)
토픽: test-from-local
Ctrl+C로 종료
--------------------------------------------------
메시지 수신:
  Key: msg-1
  Value: {'message_id': 1, 'content': 'Hello from Local PC #1', 'source': 'WSL2 Ubuntu'}
  Partition: 0, Offset: 0

메시지 수신:
  Key: msg-2
  Value: {'message_id': 2, 'content': 'Hello from Local PC #2', 'source': 'WSL2 Ubuntu'}
  Partition: 1, Offset: 0
...
```

---
## Part 6: 트러블슈팅 가이드

### 6.1 연결이 안 될 때 체크리스트

| 증상 | 원인 | 해결책 |
|------|------|--------|
| Connection timed out | Security Group에서 9093 미허용 | 인바운드 규칙 추가 |
| Connection refused | Kafka가 안 떠있음 | `docker compose ps` 확인 |
| Broker not available | ADVERTISED_LISTENERS 설정 오류 | Public IP 확인 |
| Unknown host: kafka | 내부 리스너로 접속 시도 | 9093(EXTERNAL) 포트 사용 |

### 6.2 자주 발생하는 문제

**문제 1: Kafka 컨테이너가 계속 재시작됨**
```bash
docker compose logs kafka
# → 메모리 부족 가능성 (t2.micro는 1GB만)
# → t3.small 이상 권장 또는 스왑 메모리 추가
```

**문제 2: Public IP가 변경됨 (인스턴스 재시작 시)**
```bash
# EC2 Public IP는 인스턴스 중지/시작 시 변경됩니다
# 해결 방법 1: Elastic IP 할당
# 해결 방법 2: 재시작 후 환경 변수 다시 설정
export EC2_PUBLIC_IP=$(curl -s http://169.254.169.254/latest/meta-data/public-ipv4)
docker compose down
docker compose up -d
```

**문제 3: Docker 권한 문제**
```bash
# "permission denied" 에러 발생 시
sudo usermod -aG docker ec2-user
# 재접속 필요!
exit
ssh my-ec2
```

### 6.3 디버깅 명령어

```bash
# EC2에서 Kafka 상태 확인
docker compose ps
docker compose logs -f kafka

# 로컬에서 네트워크 연결 테스트
nc -zv [EC2_PUBLIC_IP] 9093
# 또는
telnet [EC2_PUBLIC_IP] 9093

# EC2에서 환경 변수 확인
echo $EC2_PUBLIC_IP
```

---
## Part 7: 실습 완료 후 정리

### 7.1 리소스 정리 (비용 절감!)

```bash
# EC2에서 Kafka 중지
cd ~/kafka-ec2
docker compose down
```

**EC2 인스턴스 정리**:
- **중지(Stop)**: 언제든 다시 시작 가능 (EBS 비용만 발생)
- **종료(Terminate)**: 완전 삭제

### 7.2 비용 주의사항

```
Free Tier 범위:
├── t2.micro: 750시간/월 (12개월)
├── EBS: 30GB (General Purpose SSD)
└── 데이터 전송: 아웃바운드 100GB/월

실습 후 반드시 확인:
├── EC2 인스턴스 중지 또는 종료
├── Elastic IP가 있다면 해제 (미사용 시 과금!)
└── 불필요한 EBS 볼륨 삭제
```

---
## 퀴즈

### Q1. EC2의 Kafka에 외부에서 연결하려면 Security Group에서 어떤 포트를 열어야 하나요?

- A) 9092
- B) 9093
- C) 22
- D) 8080

<details>
<summary>정답 확인</summary>

**정답: B) 9093**

- 9092: 내부(Docker 네트워크) 통신용
- 9093: 외부(로컬 PC) 연결용으로 설정
- 22: SSH 접속용
- 8080: Kafka UI 웹 접근용

이 실습에서는 EXTERNAL 리스너를 9093으로 설정했으므로,
외부에서 Kafka에 연결하려면 9093 포트를 열어야 합니다.
</details>

---

### Q2. KAFKA_ADVERTISED_LISTENERS에서 EXTERNAL 리스너를 EC2 Public IP로 설정해야 하는 이유는?

<details>
<summary>정답 확인</summary>

Kafka 클라이언트가 첫 연결 후 브로커로부터 받는 "다음 연결 주소"가
ADVERTISED_LISTENERS이기 때문입니다.

만약 `kafka:9092`(내부 주소)만 설정하면:
1. 로컬 PC → EC2:9093 첫 연결 성공
2. 브로커가 "다음엔 kafka:9092로 와" 라고 알려줌
3. 로컬 PC는 `kafka`라는 호스트를 모르므로 연결 실패

EXTERNAL을 EC2 Public IP로 설정하면:
1. 로컬 PC → EC2:9093 연결
2. 브로커가 "다음엔 EC2_PUBLIC_IP:9093으로 와" 라고 알려줌
3. 로컬 PC가 계속 연결 유지 가능
</details>

---

### Q3. 실습 완료 후 비용을 아끼려면 어떻게 해야 하나요? (모두 선택)

- A) EC2 인스턴스 중지(Stop)
- B) Elastic IP가 있다면 해제
- C) Security Group 삭제
- D) 사용하지 않는 EBS 볼륨 삭제

<details>
<summary>정답 확인</summary>

**정답: A, B, D**

- **A) EC2 인스턴스 중지**: 실행 시간에 대한 과금이 멈춤
- **B) Elastic IP 해제**: 연결되지 않은 Elastic IP는 과금됨
- **D) EBS 볼륨 삭제**: 사용하지 않는 볼륨은 과금됨

C는 틀림:
- Security Group 자체는 무료이며, 삭제하면 다음 실습 때 다시 만들어야 함
</details>

---
## 과제

### 과제 1: 기본 연결 확인 (난이도: ★☆☆)

**목표**: EC2의 Kafka UI에서 토픽과 메시지 확인하기

**요구사항**:
1. 로컬에서 Producer로 메시지 5개 전송
2. `http://EC2_PUBLIC_IP:8080`에서 Kafka UI 접속
3. 토픽 목록에서 생성된 토픽 확인
4. Messages 탭에서 전송된 메시지 내용 확인

<details>
<summary>힌트</summary>

```python
# Producer에서 토픽 이름을 고유하게 설정
producer.produce(
    topic='my-name-test',  # 본인 이름으로 토픽 생성
    ...
)
```
</details>

---

### 과제 2: Key 기반 파티션 분배 확인 (난이도: ★★☆)

**목표**: 같은 Key를 가진 메시지가 같은 파티션에 저장되는지 확인

**요구사항**:
1. 3명의 사용자(user-A, user-B, user-C)가 각각 5개씩 메시지 전송
2. Key로 사용자 ID 사용
3. Kafka UI에서 각 파티션별 메시지 확인
4. 같은 사용자의 메시지가 같은 파티션에 있는지 확인

<details>
<summary>힌트</summary>

```python
users = ['user-A', 'user-B', 'user-C']
for user in users:
    for i in range(5):
        producer.produce(
            topic='user-events',
            key=user.encode('utf-8'),  # Key로 파티션 결정
            value=...
        )
```
</details>

<details>
<summary>모범답안</summary>

```python
# producer_hw2.py
from confluent_kafka import Producer
import json

EC2_PUBLIC_IP = "YOUR_EC2_IP"

def delivery_cb(err, msg):
    if not err:
        user = msg.key().decode('utf-8')
        print(f"  {user} -> Partition {msg.partition()}, Offset {msg.offset()}")

producer = Producer({
    'bootstrap.servers': f'{EC2_PUBLIC_IP}:9093',
    'client.id': 'key-test-producer'
})

users = ['user-A', 'user-B', 'user-C']

for user in users:
    print(f"\n{user}의 메시지 전송:")
    for i in range(1, 6):
        msg = {
            "user_id": user,
            "event_num": i,
            "event_type": "click"
        }
        producer.produce(
            topic='user-partition-test',
            key=user.encode('utf-8'),
            value=json.dumps(msg).encode('utf-8'),
            callback=delivery_cb
        )

producer.flush()
print("\n완료! 같은 user가 같은 파티션에 있는지 확인하세요")
```

**예상 결과**:
```
user-A의 메시지 전송:
  user-A -> Partition 0, Offset 0
  user-A -> Partition 0, Offset 1
...

user-B의 메시지 전송:
  user-B -> Partition 2, Offset 0
  user-B -> Partition 2, Offset 1
...
```
같은 user의 모든 메시지가 동일한 파티션에 저장됩니다.
</details>

---
## 핵심 요약

| 항목 | 내용 |
|------|------|
| **Security Group** | SSH(22), Kafka(9093), UI(8080) 열기 |
| **ADVERTISED_LISTENERS** | INTERNAL(내부), EXTERNAL(외부 IP) 분리 |
| **연결 주소** | 로컬 → EC2_PUBLIC_IP:9093 |
| **Kafka UI** | http://EC2_PUBLIC_IP:8080 |
| **비용 절감** | 실습 후 인스턴스 중지/종료, Elastic IP 해제 |

```
🎯 이번 교시에서 배운 것:

1. AWS에서 서비스를 배포하는 기본 흐름
   로컬 개발 → EC2 배포 → 네트워크 설정 → 외부 연결

2. 네트워크 설정의 중요성
   Security Group + ADVERTISED_LISTENERS = 외부 접근 가능

3. 클라우드 운영 감각
   비용 관리, 리소스 정리, 트러블슈팅
```

---
## 실제로 이렇게 EC2에 직접 구축하나요?

실무에서는 **관리형 서비스**를 많이 사용합니다:

| 서비스 | 설명 |
|--------|------|
| **AWS MSK** | AWS에서 제공하는 관리형 Kafka |
| **Confluent Cloud** | Kafka 개발사의 클라우드 서비스 |

**직접 클러스터를 구축하는 경우**:
- 비용 최적화가 필요한 경우
- 특수한 설정이 필요한 경우
- 학습/테스트 목적

오늘 배운 EC2 + Docker 방식은 **개념 이해와 프로토타이핑**에 적합합니다!